# Controlled MedGemma 4B Parameter-Efficient Fine-Tuning Pilot
**InteractMed-VLM: Multimodal Projector & Language LoRA Adaptation (Tier 2)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityakachhot/interactmed/blob/main/notebooks/medgemma_finetune_colab.ipynb)

### Mandatory Governance & Hardware Prerequisites:
- **Target Accelerator:** NVIDIA A100 GPU (40GB or 80GB VRAM) supporting CUDA compute capability >= 8.0 and native `bfloat16`.
- **Base Architecture:** `google/medgemma-4b-it` (Gemma 3 multimodal vision architecture: 4.3B parameters, SigLIP vision encoder, 2560 embedding dim).
- **Adaptation Hierarchy (Tier 2):** Low-Rank Adaptation (LoRA) on language decoder attention/MLP layers and multimodal projector. **Vision tower is strictly frozen** for scientific baseline attribution.
- **Split Independence:** Trained strictly on 200 patient-disjoint development cases (`train_dev`), evaluated on 50 validation cases (`val_dev`).
- **Quarantined Test Set:** The 117-case held-out test cohort (`expanded_test`) is permanently locked and must NOT be accessed during training.

## 1. Accelerator & Environment Preflight Verification
Verify that an NVIDIA GPU with compute capability >= 8.0 (Ampere/Hopper) and >= 38 GB VRAM is allocated.

In [ ]:
import torch, sys
print(f"PyTorch Version: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected! Please go to Runtime -> Change runtime type -> Hardware accelerator -> A100 GPU.")

dev_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
total_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
print(f"GPU: {dev_name}")
print(f"Compute Capability: {capability[0]}.{capability[1]}")
print(f"Total VRAM: {total_vram_gb} GB")

assert capability[0] >= 8, f"GPU capability {capability} does not support native bfloat16. An NVIDIA A100/H100 is required."
assert total_vram_gb >= 35.0, f"Insufficient VRAM ({total_vram_gb} GB). Minimum 40 GB A100 required for stable 4B LoRA training."
print("✓ Hardware accelerator verified successfully!")

## 2. Install Required Dependencies
Install `transformers >= 4.50.0` (required for Gemma 3 multimodal vision architecture) and PEFT / bitsandbytes.

In [ ]:
!pip install --upgrade --quiet "transformers>=4.50.0" peft trl bitsandbytes datasets evaluate accelerate tensorboard
import bitsandbytes as bnb
print(f"✓ bitsandbytes {bnb.__version__} successfully imported.")
# Verify 4-bit kernel on CUDA
test_t = torch.randn(64, 64, device="cuda", dtype=torch.bfloat16)
qw, qs = bnb.functional.quantize_4bit(test_t, quant_type="nf4")
deq = bnb.functional.dequantize_4bit(qw, qs, quant_type="nf4")
print("✓ 4-bit NormalFloat (NF4) quantization kernel operational on CUDA.")

## 3. Hugging Face Authentication & Gated Model Check
Authenticate using your Hugging Face access token with permissions for `google/medgemma-4b-it`.

In [ ]:
import os
from huggingface_hub import login, HfApi
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

api = HfApi()
model_id = "google/medgemma-4b-it"
info = api.model_info(model_id)
print(f"✓ Verified access to gated model: {model_id}")

## 4. Clone Repository & Verify Dataset Partitions
Check that `train_dev.jsonl` (200 cases) and `val_dev.jsonl` (50 cases) exist and verify that `expanded_test` is strictly quarantined.

In [ ]:
import os, json
from pathlib import Path
if not os.path.exists("interactmed"):
    !git clone https://github.com/adityakachhot/interactmed.git
    %cd interactmed

train_file = Path("artifacts/fine_tuning_cohort/train_dev.jsonl")
val_file = Path("artifacts/fine_tuning_cohort/val_dev.jsonl")

assert train_file.exists(), "train_dev.jsonl not found!"
assert val_file.exists(), "val_dev.jsonl not found!"

with open(train_file) as f:
    train_data = [json.loads(l) for l in f if l.strip()]
with open(val_file) as f:
    val_data = [json.loads(l) for l in f if l.strip()]

print(f"✓ Loaded {len(train_data)} training cases and {len(val_data)} validation cases.")
# Assert patient disjointness
tr_pids = set(x["patient_id"] for x in train_data)
va_pids = set(x["patient_id"] for x in val_data)
assert len(tr_pids & va_pids) == 0, "CRITICAL DATA LEAKAGE: Patient overlap between train and val!"
print("✓ Non-contamination confirmed: 0 patient overlap between train_dev and val_dev.")

## 5. Load Base Model with 4-Bit QLoRA & AutoProcessor

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

model_id = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_storage=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "right"

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
print("✓ Base model loaded successfully in 4-bit QLoRA precision.")

## 6. Configure LoRA: Freeze SigLIP Vision Tower & Target Projector + Language Decoder

In [ ]:
from peft import LoraConfig, get_peft_model

# 1. Freeze vision tower strictly for Tier 2 adaptation
frozen_vision_params = 0
for name, param in model.named_parameters():
    if "vision" in name.lower() or "siglip" in name.lower():
        param.requires_grad = False
        frozen_vision_params += param.numel()
print(f"Frozen vision encoder weights: {frozen_vision_params:,} parameters")

# 2. Configure PEFT for language attention/MLP and multimodal projector
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "multi_modal_projector.linear_1", "multi_modal_projector.linear_2"
    ],
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"],
)

peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

## 7. Data Collation with Prompt Masking
Mask prompt instruction tokens, image tokens, and padding tokens so cross-entropy loss is computed strictly over target report tokens (`OBSERVATIONS`, `IMPRESSION`, `LIMITATIONS`).

In [ ]:
from PIL import Image

def collate_fn(examples):
    texts = []
    images = []
    for ex in examples:
        with Image.open(ex["image"]) as img:
            images.append([img.convert("RGB")])
        texts.append(processor.apply_chat_template(
            ex["messages"], add_generation_prompt=False, tokenize=False
        ).strip())
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    
    if processor.tokenizer.pad_token_id is not None:
        labels[labels == processor.tokenizer.pad_token_id] = -100
    if "boi_token" in processor.tokenizer.special_tokens_map:
        boi_id = processor.tokenizer.convert_tokens_to_ids(processor.tokenizer.special_tokens_map["boi_token"])
        labels[labels == boi_id] = -100
    labels[labels == 262144] = -100
    
    for i, ex in enumerate(examples):
        user_text = processor.apply_chat_template([ex["messages"][0]], add_generation_prompt=True, tokenize=False).strip()
        prompt_len = min(len(processor.tokenizer(user_text)["input_ids"]), labels[i].shape[0])
        labels[i, :prompt_len] = -100
    
    batch["labels"] = labels
    return batch

print("✓ Custom multimodal data collator ready.")

## 8. One-Batch Overfit Sanity Check (20 Steps)
Verify optimization dynamics on 2 samples. Loss must decrease monotonically before launching full training.

In [ ]:
test_batch = collate_fn(train_data[:2])
opt = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=1e-4)
peft_model.train()

print("Executing 20-step preflight gradient descent verification:")
losses = []
for step in range(1, 21):
    opt.zero_grad()
    kwargs = {"input_ids": test_batch["input_ids"].cuda(), "labels": test_batch["labels"].cuda()}
    if "pixel_values" in test_batch and test_batch["pixel_values"] is not None:
        kwargs["pixel_values"] = test_batch["pixel_values"].cuda()
    out = peft_model(**kwargs)
    loss = out.loss
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if step % 5 == 0 or step == 1 or step == 20:
        print(f"  Step {step:02d} — Loss: {loss.item():.4f}")

assert losses[-1] < losses[0], f"Loss failed to decrease ({losses[0]:.4f} -> {losses[-1]:.4f})!"
print(f"✓ PASS: Loss dropped from {losses[0]:.4f} to {losses[-1]:.4f}. Gradients and architecture verified!")

## 9. SFTTrainer Training Configuration with Early Stopping
Configured with 3 epochs, effective batch size = 16, validation loss evaluation per epoch, and early stopping.

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback

output_ckpt_dir = "checkpoints/medgemma_lora_cloud_run_01"

training_args = SFTConfig(
    output_dir=output_ckpt_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16, # Effective batch size = 16
    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    bf16=True,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="tensorboard",
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    processing_class=processor,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print("✓ SFTTrainer configured. Ready to train.")

## 10. Execute Supervised Fine-Tuning & Export Checkpoint

In [ ]:
# Launch training
train_result = trainer.train()
print("✓ Training complete!")

# Save best model and processor
trainer.save_model(output_ckpt_dir)
processor.save_pretrained(output_ckpt_dir)

# Compute SHA256 checksums
!shasum -a 256 {output_ckpt_dir}/* > {output_ckpt_dir}/checksums.sha256
print(f"✓ Checkpoint and processor saved to {output_ckpt_dir}.")
!cat {output_ckpt_dir}/checksums.sha256

## 11. Controlled Validation Evaluation on `val_dev` (50 Cases)
Benchmark the newly trained checkpoint against Baseline Prompt v3 and Whole-Image Prompt v4.

In [ ]:
!python scripts/evaluate_finetuned_medgemma.py --split val_dev --adapter-path {output_ckpt_dir}